# 01 — EDA, chọn base/novel class, chia tập

**Input:** Kaggle Dataset BDD100K (ảnh `100k/{train,val}` + nhãn detection). **GPU:** không cần.
**Output:** `art/splits/` (gắn làm input cho các notebook sau).

**GATE:** xem bảng EDA rồi đặt `NOVEL` + `REASON`. Trong SMOKE mặc định `bus`.

In [ ]:
# === Setup (giống nhau ở mọi notebook) ===
SMOKE = True   # True: pipeline thật trên tập nhỏ. Run thật: False
REPO_URL = "https://github.com/Le-Anh-Duy/topicx-yolo-expansion.git"
COMMIT = "main"                                    # run thật: pin commit hash
import os, subprocess, sys
R = "/tmp/repo"
if not os.path.exists(R):
    subprocess.run(["git", "clone", "-q", REPO_URL, R], check=True)
subprocess.run(["git", "-C", R, "fetch", "-q"], check=True)
subprocess.run(["git", "-C", R, "checkout", "-q", COMMIT], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", f"{R}/requirements.txt"], check=True)
sys.path.insert(0, f"{R}/src")
import numpy as np, pandas as pd
from topicx import common as C
cfg = C.load_config(SMOKE)
print(cfg["art"])

In [ ]:
STAGE = "splits"
# Test protocol (CPU, vài giây). Lỗi thì dừng notebook.
subprocess.run([sys.executable, "-m", "pytest", "-q", f"{R}/tests"], check=True)
C.env_report(cfg, STAGE)

In [ ]:
from topicx import data as D
labels, image_dirs = D.locate_bdd(C.INPUT)
print(labels, image_dirs, sep="\n")
assert set(labels) == {"train", "val"} and set(image_dirs) == {"train", "val"}, "gắn dataset BDD100K có images/100k và nhãn det"
parts = [D.load_labels(labels[s], s, cfg["alias"], cfg["classes"]) for s in ("train", "val")]
images = pd.concat([p[0] for p in parts], ignore_index=True)
boxes = pd.concat([p[1] for p in parts], ignore_index=True)
assert images.image.is_unique
paths = D.image_paths(images, image_dirs)
missing = {i for i in images.image if not paths[i].exists()}
print(f"{len(images)} ảnh có nhãn, {len(missing)} không có file ảnh -> bỏ")
images = images[~images.image.isin(missing)].reset_index(drop=True)
boxes = boxes[boxes.image.isin(set(images.image))].reset_index(drop=True)
from PIL import Image
sizes = {Image.open(paths[i]).size for i in images.image.sample(200, random_state=0)}
assert sizes == {tuple(cfg["img_wh"])}, sizes

## EDA
`removal_cost`: tỉ lệ instance mỗi class bị mất khỏi base train nếu loại mọi ảnh chứa novel candidate (co-occurrence).

In [ ]:
stats = D.class_stats(images, boxes)
display(stats)
display(D.removal_cost(boxes, cfg["novel_candidates"]))
display(images.timeofday.value_counts())
n = len(images)
proj = pd.DataFrame({c: {"pool_pos_images": stats.n_images[c] / n * cfg["sizes"]["pool"],
                         "test_instances": stats.n_instances[c] / n * cfg["sizes"]["test"],
                         "random_K_pos_images": {k: round(stats.n_images[c] / n * k, 1) for k in cfg["ks"]}}
                     for c in cfg["novel_candidates"]}).T
display(proj)   # ước tính theo tỉ lệ toàn tập

In [ ]:
# GATE: chọn novel class sau khi xem bảng (đủ instance ở test, removal_cost thấp với base classes, pool có đủ positive)
NOVEL = "bus" if SMOKE else None
REASON = "smoke" if SMOKE else ""
assert NOVEL in cfg["novel_candidates"] and REASON, "đặt NOVEL và REASON"

In [ ]:
has_novel = set(boxes.image[boxes.cls == NOVEL])
splits = D.make_splits(images, has_novel, cfg["sizes"], cfg["seed_split"])
report = D.check_splits(splits, images, has_novel)
dups = D.md5_duplicates(splits, paths) if cfg["md5_check"] else None
assert not dups, dups[:5]
base_names, names = D.class_names(cfg["classes"], NOVEL)
perm = np.random.default_rng(cfg["seed_split"]).permutation(len(splits["base"]))
base_sorted = sorted(splits["base"])
replay = [base_sorted[i] for i in perm[:cfg["replay"]]]
control = [base_sorted[i] for i in perm[cfg["replay"]:cfg["replay"] + max(cfg["ks"])]]
assert len(control) == max(cfg["ks"]), "base train quá nhỏ cho replay + control"
test_novel = int(boxes[boxes.image.isin(set(splits["test"])) & (boxes.cls == NOVEL)].shape[0])
print(pd.DataFrame(report).T, f"\nnovel instances ở test: {test_novel}")
if test_novel < 300 and not SMOKE:
    print("CẢNH BÁO: < 300 novel instances ở test -> AP novel dao động mạnh giữa seed")

In [ ]:
for s, ids in splits.items():
    C.write_ids(ids, C.out(cfg, f"splits/{s}_ids.txt"))
C.write_ids(replay, C.out(cfg, "splits/replay_ids.txt"))
C.write_ids(control, C.out(cfg, "splits/control_ids.txt"))
all_ids = {i for v in splits.values() for i in v}
pool = set(splits["pool"])
images[images.image.isin(all_ids)][["image", "src"]].to_csv(C.out(cfg, "splits/image_src.csv"), index=False)
pub = all_ids - pool
images[images.image.isin(pub)].to_csv(C.out(cfg, "splits/public_images.csv"), index=False)
boxes[boxes.image.isin(pub)].to_csv(C.out(cfg, "splits/public_boxes.csv"), index=False)
# Nhãn + thuộc tính ảnh của pool chỉ nằm trong oracle/
images[images.image.isin(pool)].to_csv(C.out(cfg, "splits/oracle/pool_images.csv"), index=False)
boxes[boxes.image.isin(pool)].to_csv(C.out(cfg, "splits/oracle/pool_boxes.csv"), index=False)
C.write_ids(splits["pool"], C.out(cfg, "splits/oracle/pool_ids.txt"))
C.dump({"novel": NOVEL, "reason": REASON, "base_names": base_names, "names": names, "sizes": cfg["sizes"],
        "seed_split": cfg["seed_split"], "report": report, "test_novel_instances": test_novel,
        "md5_checked": cfg["md5_check"], "n_missing_files": len(missing), "class_stats": stats.to_dict()},
       C.out(cfg, "splits/meta.json"))
print("xong -> Save Version, gắn output này làm input cho NB2–NB5")